# Notebook title

description here

### Import Libraries

fill this out later

In [1]:
import pandas as pd
import numpy as np
from nba_api.stats.endpoints import scheduleleaguev2

pd.set_option('display.max_columns', None)

SEASON = "2025-26"
DATA = "../data/2025_raw_box_score_team_stats.csv"


### Consolidate games 

fill this out later

In [2]:
def game_consolidation(season_data):
    '''
    Fill this out later
    '''
    df = pd.read_csv(season_data)
    df = df.drop(columns=[
                "TEAM_ID", "SEASON_ID", "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT",
                "FTM", "FTA", "FT_PCT", "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PLUS_MINUS"
            ])

    home = df[df["MATCHUP"].str.contains("vs.")]
    away = df[df["MATCHUP"].str.contains("@")]
    games = home.merge(away, on="GAME_ID", suffixes=("_HOME", "_AWAY"))
    games = games.rename(columns={"MATCHUP_AWAY": 'MATCHUP'})
    games['WINNER'] = np.where(games['WL_HOME'] == "W", games['TEAM_NAME_HOME'], games['TEAM_NAME_AWAY'])

    games = games[[
            "GAME_ID", "GAME_DATE_HOME", "MATCHUP", 
            "TEAM_ABBREVIATION_AWAY", "TEAM_NAME_AWAY", "PTS_AWAY",
            "TEAM_ABBREVIATION_HOME", "TEAM_NAME_HOME", "PTS_HOME", "WINNER"
        ]]

    return games


test = game_consolidation(DATA)
test


,GAME_ID,GAME_DATE_HOME,MATCHUP,TEAM_ABBREVIATION_AWAY,TEAM_NAME_AWAY,PTS_AWAY,TEAM_ABBREVIATION_HOME,TEAM_NAME_HOME,PTS_HOME,WINNER
0,22500002,2025-10-21,GSW @ LAL,GSW,Golden State Warriors,119,LAL,Los Angeles Lakers,109,Golden State Warriors
1,22500001,2025-10-21,HOU @ OKC,HOU,Houston Rockets,124,OKC,Oklahoma City Thunder,125,Oklahoma City Thunder
2,22500086,2025-10-22,WAS @ MIL,WAS,Washington Wizards,120,MIL,Milwaukee Bucks,133,Milwaukee Bucks
3,22500003,2025-10-22,CLE @ NYK,CLE,Cleveland Cavaliers,111,NYK,New York Knicks,119,New York Knicks
4,22500081,2025-10-22,MIA @ ORL,MIA,Miami Heat,121,ORL,Orlando Magic,125,Orlando Magic
...,...,...,...,...,...,...,...,...,...,...
1220,22501193,2026-04-12,CHI @ DAL,CHI,Chicago Bulls,128,DAL,Dallas Mavericks,149,Dallas Mavericks
1221,22501200,2026-04-12,SAC @ POR,SAC,Sacramento Kings,110,POR,Portland Trail Blazers,122,Portland Trail Blazers
1222,22501197,2026-04-12,DEN @ SAS,DEN,Denver Nuggets,128,SAS,San Antonio Spurs,118,Denver Nuggets
1223,22501196,2026-04-12,PHX @ OKC,PHX,Phoenix Suns,135,OKC,Oklahoma City Thunder,103,Phoenix Suns


### Get the game start times

fill this out later


In [4]:
def nba_game_start_times(year):
    '''
    Fill this out later
    '''
    sched = scheduleleaguev2.ScheduleLeagueV2(season=year)
    df = sched.season_games.get_data_frame()

    df = df[["gameId", "gameDate", "gameDateTimeUTC","gameDateTimeEst",
            "gameStatusText", "homeTeam_teamTricode", "awayTeam_teamTricode", "gameLabel"]]

    df = df.rename(columns={"gameId": 'GAME_ID'})
    df["GAME_ID"] = pd.to_numeric(df["GAME_ID"], errors="coerce").astype("Int64")
    games_to_keep = ['', 'Emirates NBA Cup', 'NBA Mexico City Game', 'NBA Berlin Game', 'NBA London Game', 'AWS NBA Rivals Week','NBA Pioneers Classic']
    df_filtered = df.loc[df['gameLabel'].isin(games_to_keep)]

    return df_filtered

test2 = nba_game_start_times(SEASON)
test2


,GAME_ID,gameDate,gameDateTimeUTC,gameDateTimeEst,gameStatusText,homeTeam_teamTricode,awayTeam_teamTricode,gameLabel
71,22500001,10/21/2025 00:00:00,2025-10-21T23:30:00Z,2025-10-21T19:30:00Z,Final/OT2,OKC,HOU,
72,22500002,10/21/2025 00:00:00,2025-10-22T02:00:00Z,2025-10-21T22:00:00Z,Final,LAL,GSW,
73,22500003,10/22/2025 00:00:00,2025-10-22T23:00:00Z,2025-10-22T19:00:00Z,Final,NYK,CLE,
74,22500004,10/22/2025 00:00:00,2025-10-23T01:30:00Z,2025-10-22T21:30:00Z,Final,DAL,SAS,
75,22500080,10/22/2025 00:00:00,2025-10-22T23:00:00Z,2025-10-22T19:00:00Z,Final,CHA,BKN,
...,...,...,...,...,...,...,...,...
1304,22501196,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,OKC,PHX,
1305,22501197,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,SAS,DEN,
1306,22501198,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,LAL,UTA,
1307,22501199,04/12/2026 00:00:00,2026-04-13T00:30:00Z,2026-04-12T20:30:00Z,Final,LAC,GSW,


### TO DO

1. Figure out why the missing games: both teams considered 'AWAY' beacuse they are playing a different country.
    a. Dropping those games for now but can easily add them later if we want.

2. Merge df's and use UTC timestamp for fetching kalshi data

upload Kalshi functions

Fetch Kalshi data